# CFPB Complaint Routing — Phase 3: Fine-tuned Transformer Router
**Question:** does a pretrained transformer beat the TF-IDF + Linear SVM baseline from Phase 2 (test macro-F1 **0.781**), and does it rely less on company names?

Plan:
1. Same data, same tidy-up, same **time-based split** as Phase 2
2. Re-fit the Phase 2 baseline on the *same* training window so the comparison is apples-to-apples
3. Fine-tune **DistilRoBERTa** (82M params) with class-weighted loss, mixed precision and length-bucketed batches
4. Compare on the untouched test window — overall, per team, and on the hardest confusion (debt collection vs credit reporting)
5. **Shortcut test:** mask company names in the test narratives and see which model's accuracy collapses
6. Confidence-based routing: how much can be auto-routed at ≥95% accuracy?

Runs on **Kaggle Notebooks** with a **GPU (T4 or P100)** and Internet on. Self-contained: it rebuilds the Phase 1 sample from the archived Hugging Face snapshot with the same seed.

## 1. Setup

In [ ]:
import os, re, time, json, gc, math, random, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 300)
SEED = 42
OUT_DIR = "/kaggle/working"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| GPUs:", torch.cuda.device_count(),
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU - turn on an accelerator in Settings")

## 2. Data — identical to Phase 2

In [ ]:
import glob
cached = glob.glob("/kaggle/input/**/cfpb_sample.parquet", recursive=True)
if cached:
    df = pd.read_parquet(cached[0]); print("loaded cached sample:", cached[0])
else:
    from huggingface_hub import hf_hub_download
    import pyarrow.parquet as pq
    PQ = hf_hub_download("vivekkopthsd/cfpb-consumer-complaints", "complaints.parquet",
                         repo_type="dataset", local_dir="/kaggle/temp")
    pf = pq.ParquetFile(PQ)
    COLS = ["date_received", "product", "sub_product", "issue", "sub_issue", "narrative",
            "company", "state", "company_response", "timely_response", "complaint_id"]
    START = pd.Timestamp("2020-01-01"); parts = []
    for batch in pf.iter_batches(batch_size=250_000, columns=COLS):
        chunk = batch.to_pandas()
        chunk["date_received"] = pd.to_datetime(chunk["date_received"], errors="coerce")
        chunk = chunk[chunk["narrative"].notna() & (chunk["narrative"].str.len() > 0)]
        chunk = chunk[chunk["date_received"] >= START]
        parts.append(chunk.sample(frac=0.15, random_state=SEED))
    df = pd.concat(parts, ignore_index=True); del parts; gc.collect()

    PRODUCT_MAP = {
        "Credit reporting, credit repair services, or other personal consumer reports": "Credit reporting",
        "Credit reporting or other personal consumer reports": "Credit reporting",
        "Credit reporting": "Credit reporting",
        "Debt collection": "Debt collection", "Debt or credit management": "Debt collection",
        "Credit card or prepaid card": "Credit card", "Credit card": "Credit card",
        "Prepaid card": "Prepaid / money transfer",
        "Money transfer, virtual currency, or money service": "Prepaid / money transfer",
        "Money transfers": "Prepaid / money transfer", "Virtual currency": "Prepaid / money transfer",
        "Checking or savings account": "Bank account", "Bank account or service": "Bank account",
        "Mortgage": "Mortgage", "Vehicle loan or lease": "Vehicle loan", "Consumer Loan": "Vehicle loan",
        "Student loan": "Student loan",
        "Payday loan, title loan, or personal loan": "Personal / payday loan",
        "Payday loan, title loan, personal loan, or advance loan": "Personal / payday loan",
        "Payday loan": "Personal / payday loan",
    }
    df["team"] = df["product"].map(PRODUCT_MAP).fillna("Other")

    def clean_text(s):
        s = re.sub(r"X{2,}/X{2,}/(X{2,}|\d{2,4})", " ", s)
        s = re.sub(r"\bX{2,}\b", " ", s)
        s = re.sub(r"\{\$[\d,.]+\}", " <amount> ", s)
        return re.sub(r"\s+", " ", s).strip()
    df["text"] = df["narrative"].map(clean_text)
    df["n_words"] = df["text"].str.split().str.len()
    df = df.drop_duplicates(subset="text")
    df = df[df["n_words"] >= 10].drop(columns="narrative").reset_index(drop=True)
    df.to_parquet(f"{OUT_DIR}/cfpb_sample.parquet", index=False)
print(df.shape)
print(df["team"].value_counts())

In [ ]:
def tidy(s):
    s = re.sub(r"/?\s*/?year>", " ", s)            # leftover 'XX/XX/year>' fragments
    s = re.sub(r"(?:\s*[,/;:]\s*){2,}", ", ", s)     # runs of punctuation left by removed redactions
    s = re.sub(r"\(\s*[,\s]*\)", " ", s)             # empty parentheses
    s = re.sub(r"^\s*(On|on)\s*,\s*", "", s)          # 'On , I ...' -> 'I ...'
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

before = df["text"].iloc[0]
df["text"] = df["text"].map(tidy)
print("BEFORE:", before[:200]); print("AFTER: ", df["text"].iloc[0][:200])
n0 = len(df)
df = df.drop_duplicates(subset="text").reset_index(drop=True)
print(f"\nrows: {n0:,} -> {len(df):,} after re-dedup")

In [ ]:
TRAIN_END, VAL_END = "2024-12-31", "2025-06-30"
train = df[df.date_received <= TRAIN_END]
val   = df[(df.date_received > TRAIN_END) & (df.date_received <= VAL_END)]
test  = df[df.date_received > VAL_END]
TEAMS = sorted(df["team"].unique())

split_dist = pd.DataFrame({name: d["team"].value_counts(normalize=True) for name, d in
                           [("train", train), ("val", val), ("test", test)]}).loc[TEAMS]
print({name: len(d) for name, d in [("train", train), ("val", val), ("test", test)]})
(split_dist * 100).round(1)

## 3. Company-name masking (for the shortcut test)
Phase 2 showed the baseline's top features include company names (Equifax, Chime, Synchrony, Coinbase, Mohela, ...). That works, but it is a shortcut: a router should understand the *problem*, not just *who* it's about.
We build a list of brand tokens from the most frequent companies plus well-known brands, and replace them with `[COMPANY]`. Models are trained on normal text; masking is applied **only at test time** to measure how much each model depends on names.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
# 1) curated brands: unambiguous names are masked case-insensitively ...
BRANDS_CI = """equifax experian transunion lexisnexis chime synchrony barclays comenity citi citibank citigroup mastercard amex
paypal venmo zelle cashapp coinbase binance kraken robinhood affirm klarna upstart netcredit greensky acima mohela navient
nelnet aidvantage edfinancial fedloan sallie santander bridgecrest carvana toyota honda bmw fargo jpmorgan pnc truist
huntington ocwen loancare newrez pennymac caliber resurgent lvnv cavalry webbank goldman sofi lendingclub usaa""".split()
# ... brands that are also everyday words ("I discovered", "chase a payment") only when capitalised
BRANDS_CS = "Discover Chase Square Block Wise Apple Rocket Regions Ally Marcus Cooper Midland Jefferson Gemini Visa Navy".split()

# 2) data-driven: words from the top-150 company names, kept only if they are *specific* to that company,
#    i.e. >=40% of training narratives mentioning the word are complaints against a company whose name contains it
GENERIC = set("""bank banks national financial services service group holdings holding corporation company credit card cards
capital trust union federal mortgage loan loans home lending servicing solutions information management partners associates
recovery collection collections systems consumer technologies technology payments payment fund funding auto north south west
east city community savings america american first account accounts acceptance association attorney automobile asset
assets analytics avenue bancorp""".split())
cand = sorted({w for comp in df["company"].value_counts().head(150).index for w in re.findall(r"[a-z]+", comp.lower())
               if len(w) > 3 and w not in GENERIC and w not in ENGLISH_STOP_WORDS})
samp = train.sample(min(60_000, len(train)), random_state=SEED)
M = CountVectorizer(vocabulary=cand, binary=True).fit_transform(samp["text"]).tocsc()
comp_low = samp["company"].str.lower().values
specific = []
for j, w in enumerate(cand):
    rows = M[:, j].indices
    if len(rows) >= 20 and np.mean([w in comp_low[r] for r in rows]) >= 0.4:
        specific.append(w)
ci_tokens = sorted(set(BRANDS_CI) | set(specific))
print(f"{len(cand)} candidate words from company names -> {len(specific)} kept as company-specific:")
print(", ".join(specific))
pat_ci = re.compile(r"\b(" + "|".join(sorted(map(re.escape, ci_tokens), key=len, reverse=True)) + r"|cash app)\b", re.I)
pat_cs = re.compile(r"\b(" + "|".join(BRANDS_CS) + r")\b")
mask_companies = lambda s: pat_cs.sub("[COMPANY]", pat_ci.sub("[COMPANY]", s))

test_masked_text = test["text"].map(mask_companies)
changed = (test_masked_text != test["text"]).values
ex = test.loc[changed, "text"].iloc[0] if changed.any() else test["text"].iloc[0]
print("\nBEFORE:", ex[:300]); print("AFTER: ", mask_companies(ex)[:300])
print(f"\ntest narratives containing at least one masked name: {changed.mean():.1%}")


## 4. Baseline re-fit on the same training window
Phase 2's headline number (0.781) came from a model refit on train **+ validation**. Here both models see **only the 2020–2024 training window**, so the comparison is fair.

In [ ]:
t0 = time.time()
TFIDF_KW = dict(ngram_range=(1, 2), min_df=10, max_df=0.9, sublinear_tf=True,
                max_features=200_000, strip_accents="unicode", dtype=np.float32)
vec = TfidfVectorizer(**TFIDF_KW)
X_tr = vec.fit_transform(train["text"])
svm = LinearSVC(C=0.25, class_weight="balanced").fit(X_tr, train["team"])
del X_tr; gc.collect()
y_va, y_te = val["team"].values, test["team"].values
svm_val  = svm.predict(vec.transform(val["text"]))
svm_scores = svm.decision_function(vec.transform(test["text"]))
svm_test = svm.classes_[svm_scores.argmax(1)]
svm_mask = svm.predict(vec.transform(test_masked_text))
print(f"baseline fit+predict {time.time()-t0:.0f}s")
for name, y, p in [("val", y_va, svm_val), ("test", y_te, svm_test), ("test (names masked)", y_te, svm_mask)]:
    print(f"TF-IDF+SVM {name:20s} macro-F1 {f1_score(y, p, average='macro'):.3f}  acc {accuracy_score(y, p):.3f}")

## 5. Tokenise for the transformer
- **DistilRoBERTa** keeps casing and punctuation and was pretrained on web text, a good fit for informal complaint narratives
- `MAX_LEN = 256` tokens covers the median complaint (~126 words) comfortably; ~10% of long narratives are truncated (Phase 1 finding)

In [ ]:
MODEL_NAME, MAX_LEN = "distilroberta-base", 256
TEAMS = sorted(df["team"].unique()); label2id = {t: i for i, t in enumerate(TEAMS)}
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(texts, bs=4096):
    ids = []
    for i in range(0, len(texts), bs):
        enc = tok(list(texts[i:i + bs]), truncation=True, max_length=MAX_LEN, add_special_tokens=True)
        ids.extend(np.asarray(x, dtype=np.int32) for x in enc["input_ids"])
    return ids

t0 = time.time()
ids_tr, ids_va = encode(train["text"].values), encode(val["text"].values)
ids_te, ids_tm = encode(test["text"].values), encode(test_masked_text.values)
lab_tr = train["team"].map(label2id).values
lens = np.array([len(x) for x in ids_tr])
print(f"tokenised in {time.time()-t0:.0f}s | median len {np.median(lens):.0f} | truncated at {MAX_LEN}: {(lens >= MAX_LEN).mean():.1%}")

## 6. Fine-tuning loop
Written as a plain PyTorch loop so every step is visible:
- **class-weighted cross-entropy** (weights ∝ 1/√frequency) — softer than fully balanced weights, so the 60% credit-reporting class isn't over-penalised
- **length-bucketed batches**: similar-length narratives are batched together, cutting padding (and GPU time) substantially
- **mixed precision** (fp16), AdamW, linear warm-up/decay, gradient clipping
- evaluate on the validation window after every epoch and keep the best checkpoint

In [ ]:
PAD = tok.pad_token_id
def batches(ids, bs, shuffle):
    idx = np.arange(len(ids))
    if shuffle:
        np.random.shuffle(idx)
        chunks = [idx[i:i + bs * 50] for i in range(0, len(idx), bs * 50)]   # sort within chunks of 50 batches
        idx = np.concatenate([c[np.argsort([len(ids[j]) for j in c])] for c in chunks])
        out = [idx[i:i + bs] for i in range(0, len(idx), bs)]; random.shuffle(out); return out
    idx = idx[np.argsort([len(x) for x in ids])]                            # inference: fully sorted
    return [idx[i:i + bs] for i in range(0, len(idx), bs)]

def collate(ids, b):
    m = max(len(ids[j]) for j in b)
    x = np.full((len(b), m), PAD, dtype=np.int64); att = np.zeros((len(b), m), dtype=np.int64)
    for r, j in enumerate(b):
        x[r, :len(ids[j])] = ids[j]; att[r, :len(ids[j])] = 1
    return torch.from_numpy(x).to(DEVICE), torch.from_numpy(att).to(DEVICE)

@torch.no_grad()
def predict_proba(model, ids, bs=256):
    model.eval(); out = np.zeros((len(ids), len(TEAMS)), dtype=np.float32)
    for b in batches(ids, bs, shuffle=False):
        x, att = collate(ids, b)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            logits = model(input_ids=x, attention_mask=att).logits
        out[b] = torch.softmax(logits.float(), -1).cpu().numpy()
    return out

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(TEAMS), id2label=dict(enumerate(TEAMS)), label2id=label2id).to(DEVICE)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
core = model.module if isinstance(model, nn.DataParallel) else model

EPOCHS, BS, LR = 2, 32, 3e-5
freq = np.bincount(lab_tr, minlength=len(TEAMS)) / len(lab_tr)
w = 1 / np.sqrt(freq); w = w / (w * freq).sum()
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE))
print("class weights:", dict(zip(TEAMS, w.round(2))))

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps_per_epoch = math.ceil(len(ids_tr) / BS)
sched = get_linear_schedule_with_warmup(opt, int(0.06 * EPOCHS * steps_per_epoch), EPOCHS * steps_per_epoch)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
history, best_f1, best_state = [], -1, None
lab_tr_t = torch.tensor(lab_tr, dtype=torch.long)

for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); run_loss = 0
    for step, b in enumerate(batches(ids_tr, BS, shuffle=True), 1):
        x, att = collate(ids_tr, b); y = lab_tr_t[b].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            loss = loss_fn(model(input_ids=x, attention_mask=att).logits.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        run_loss += loss.item()
        if step % 1000 == 0:
            print(f"  epoch {epoch} step {step}/{steps_per_epoch} loss {run_loss/step:.4f} ({time.time()-t0:.0f}s)")
    p_val = predict_proba(model, ids_va).argmax(1)
    f1 = f1_score(val["team"].map(label2id), p_val, average="macro")
    history.append({"epoch": epoch, "train_loss": run_loss / step, "val_macro_f1": f1, "minutes": (time.time() - t0) / 60})
    print(history[-1])
    if f1 > best_f1:
        best_f1 = f1; best_state = {k: v.detach().cpu().clone() for k, v in core.state_dict().items()}
core.load_state_dict(best_state)
pd.DataFrame(history)

## 7. Test-window comparison

In [ ]:
P_te = predict_proba(model, ids_te); P_tm = predict_proba(model, ids_tm)
tr_test = np.array(TEAMS)[P_te.argmax(1)]; tr_mask = np.array(TEAMS)[P_tm.argmax(1)]
tr_val = np.array(TEAMS)[predict_proba(model, ids_va).argmax(1)]

def row(name, y, p): return {"model": name, "macro_f1": f1_score(y, p, average="macro"),
                             "weighted_f1": f1_score(y, p, average="weighted"), "accuracy": accuracy_score(y, p)}
comp = pd.DataFrame([
    row("TF-IDF + SVM | val", y_va, svm_val), row("DistilRoBERTa | val", y_va, tr_val),
    row("TF-IDF + SVM | test", y_te, svm_test), row("DistilRoBERTa | test", y_te, tr_test),
    row("TF-IDF + SVM | test, names masked", y_te, svm_mask), row("DistilRoBERTa | test, names masked", y_te, tr_mask),
]).set_index("model").round(3)
comp

In [ ]:
per_team = pd.DataFrame({
    "TF-IDF + SVM": f1_score(y_te, svm_test, average=None, labels=TEAMS),
    "DistilRoBERTa": f1_score(y_te, tr_test, average=None, labels=TEAMS),
    "test support": pd.Series(y_te).value_counts().reindex(TEAMS).values}, index=TEAMS)
per_team["gain"] = per_team["DistilRoBERTa"] - per_team["TF-IDF + SVM"]
fig, ax = plt.subplots(figsize=(9, 5))
per_team[["TF-IDF + SVM", "DistilRoBERTa"]].sort_values("DistilRoBERTa").plot.barh(ax=ax, color=["#9AA5B1", "#2F6FDE"])
ax.set(title="Per-team F1 on the test window", xlabel="F1"); ax.set_xlim(0, 1); plt.tight_layout(); plt.show()
print(classification_report(y_te, tr_test, digits=3))
per_team.round(3)

In [ ]:
cm = confusion_matrix(y_te, tr_test, labels=TEAMS, normalize="true")
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=TEAMS, yticklabels=TEAMS, ax=ax, cbar=False)
ax.set(title="DistilRoBERTa — test confusion matrix (row-normalised)", xlabel="predicted", ylabel="true")
plt.xticks(rotation=40, ha="right"); plt.tight_layout(); plt.show()
pair = lambda p, a, b: int(((y_te == a) & (p == b)).sum())
print("Debt collection -> Credit reporting errors:  SVM", pair(svm_test, "Debt collection", "Credit reporting"),
      "| DistilRoBERTa", pair(tr_test, "Debt collection", "Credit reporting"))
print("Prepaid/money transfer -> Bank account errors: SVM", pair(svm_test, "Prepaid / money transfer", "Bank account"),
      "| DistilRoBERTa", pair(tr_test, "Prepaid / money transfer", "Bank account"))

## 8. Shortcut test — how much does each model rely on company names?
A large drop when names are masked means the model was leaning on *who* rather than *what*.

In [ ]:
drop = pd.DataFrame({
    "test macro-F1": [comp.loc["TF-IDF + SVM | test", "macro_f1"], comp.loc["DistilRoBERTa | test", "macro_f1"]],
    "names masked":  [comp.loc["TF-IDF + SVM | test, names masked", "macro_f1"], comp.loc["DistilRoBERTa | test, names masked", "macro_f1"]],
}, index=["TF-IDF + SVM", "DistilRoBERTa"])
drop["drop"] = drop["test macro-F1"] - drop["names masked"]
drop["relative drop"] = (drop["drop"] / drop["test macro-F1"]).map("{:.1%}".format)
masked_rows = (test_masked_text != test["text"]).values
print(f"On the {masked_rows.mean():.0%} of test complaints that actually contained a name:")
for name, a, b in [("TF-IDF + SVM", svm_test, svm_mask), ("DistilRoBERTa", tr_test, tr_mask)]:
    print(f"  {name:14s} accuracy {accuracy_score(y_te[masked_rows], a[masked_rows]):.3f} -> "
          f"{accuracy_score(y_te[masked_rows], b[masked_rows]):.3f} with names masked")
drop.round(3)

## 9. Confidence-based routing
DistilRoBERTa gives calibrated-ish probabilities; the SVM gives a decision margin. For each, find the largest share of complaints we can auto-route while keeping ≥95% accuracy on that share.

In [ ]:
def coverage_curve(conf, correct):
    rows = []
    for q in np.linspace(0, 0.95, 39):
        keep = conf >= np.quantile(conf, q)
        rows.append({"coverage": keep.mean(), "accuracy": correct[keep].mean()})
    return pd.DataFrame(rows)
s = np.sort(svm_scores, 1)
curves = {"TF-IDF + SVM (margin)": coverage_curve(s[:, -1] - s[:, -2], svm_test == y_te),
          "DistilRoBERTa (max prob)": coverage_curve(P_te.max(1), tr_test == y_te)}
fig, ax = plt.subplots(figsize=(7, 4))
for name, c in curves.items(): ax.plot(c.coverage, c.accuracy, marker="o", ms=3, label=name)
ax.axhline(0.95, ls="--", c="grey", lw=1); ax.invert_xaxis(); ax.legend()
ax.set(title="Auto-routing trade-off on the test window", xlabel="share auto-routed", ylabel="accuracy on auto-routed share")
plt.show()
routing = {n: float(c[c.accuracy >= 0.95].coverage.max()) for n, c in curves.items()}
print({k: f"{v:.0%} auto-routed at >=95% accuracy" for k, v in routing.items()})

## 10. Where the transformer fixed the baseline's mistakes

In [ ]:
fixed = test.assign(svm=svm_test, tr=tr_test)
fixed = fixed[(fixed.svm != fixed.team) & (fixed.tr == fixed.team)]
print(f"{len(fixed):,} test complaints fixed by the transformer; "
      f"{int(((svm_test == y_te) & (tr_test != y_te)).sum()):,} newly broken")
for _, r in fixed.sample(min(3, len(fixed)), random_state=SEED).iterrows():
    print(f"\n--- true: {r.team} | SVM said: {r.svm}\n{r.text[:350]}")

## 11. Save the model and metrics

In [ ]:
save_dir = f"{OUT_DIR}/distilroberta-cfpb-router"
core.save_pretrained(save_dir); tok.save_pretrained(save_dir)
metrics = {"phase": 3, "model": MODEL_NAME, "max_len": MAX_LEN, "epochs": EPOCHS, "lr": LR, "batch_size": BS,
           "train_window": "2020-01-01..2024-12-31", "val_window": "2025-01..2025-06", "test_window": "2025-07..2026-07",
           "history": history, "comparison": comp.reset_index().to_dict(orient="records"),
           "per_team_f1": per_team.round(4).reset_index().to_dict(orient="records"),
           "auto_route_coverage_at_95": routing}
json.dump(metrics, open(f"{OUT_DIR}/phase3_metrics.json", "w"), indent=2, default=float)
print(json.dumps(metrics["comparison"], indent=1, default=float)[:1500])

In [ ]:
def route(narrative: str):
    p = predict_proba(model, encode([tidy(narrative)]))[0]; o = p.argsort()[::-1]
    return {"team": TEAMS[o[0]], "confidence": round(float(p[o[0]]), 3), "runner_up": TEAMS[o[1]]}
route("A collection agency keeps calling me about a debt I already paid, and now it shows up on my credit report.")

## Findings & next steps
*(summarised in the README after the run)*
- **Phase 4:** trend detection — anomaly detection on monthly complaint volumes per team / issue / company, plus topic modelling for emerging issues.